In [9]:
import torch, numpy as np
DATA_PATH = "C:/Users/SAGAR/OneDrive/Desktop/dl_genai/dl-genai-project/data/train.csv"   # <-- your path
OPTS = ["A","B","C","D","E"]
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cpu | cuda: False


In [10]:
try:
    from datasets import load_dataset
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets"])
    from datasets import load_dataset
ds = load_dataset("csv", data_files=DATA_PATH)["train"]
def add_combined(ex):
    ex["combined_text"] = ex["prompt"] + " " + ex["A"]
    return ex
ds = ds.map(add_combined)
print("ANSWER Q1:", len(ds[51]["combined_text"]))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

ANSWER Q1: 614


In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
print("ANSWER Q2:", tok.vocab_size)     
print("ANSWER Q3:", tok.sep_token_id)    

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ANSWER Q2: 30522
ANSWER Q3: 102


In [ ]:

prompts = [str(p) for p in ds["prompt"]]

enc = tok(prompts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
print("ANSWER Q4:", enc["input_ids"].shape)   

ANSWER Q4: torch.Size([2000, 128])


In [ ]:
print("ANSWER Q5:", 768 // 12)   

ANSWER Q5: 64


In [15]:
from transformers import AutoModel
model = AutoModel.from_pretrained("bert-base-uncased").eval()
inp0 = tok(ds[0]["prompt"], return_tensors="pt")
with torch.no_grad():
    out0 = model(**inp0)
print("ANSWER Q6:", out0.last_hidden_state.shape)
cls_vec = out0.last_hidden_state[0, 0, :]
print("ANSWER Q7:", round(cls_vec[:5].sum().item(), 4))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ANSWER Q6: torch.Size([1, 31, 768])
ANSWER Q7: -1.2001


In [16]:
model_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True).eval()
inp = tok("Light-ion fusion is a technique.", return_tensors="pt")
with torch.no_grad():
    out = model_attn(**inp)
tokens = tok.convert_ids_to_tokens(inp["input_ids"][0])
print("tokens:", tokens)
fusion_idx = tokens.index("fusion")
att = out.attentions[-1][0, 0]
print("ANSWER Q8:", round(att[0, fusion_idx].item(), 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
ANSWER Q8: 0.1025


In [18]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer, util
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
p_emb = st.encode(ds[0]["prompt"]); b_emb = st.encode(ds[0]["B"])
print("ANSWER Q9:", round(util.cos_sim(p_emb, b_emb).item(), 4))


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ANSWER Q9: 0.7658


In [ ]:
answers = ds["answer"]
prompts = ds["prompt"]

# ---- MiniLM ranking ----
p_all = st.encode(prompts, convert_to_numpy=True, batch_size=64, show_progress_bar=True)
o_all = {o: st.encode(ds[o], convert_to_numpy=True, batch_size=64) for o in OPTS}
def cos_rows(a, b):
    na = np.linalg.norm(a,axis=1); nb = np.linalg.norm(b,axis=1)
    return (a*b).sum(1)/(na*nb+1e-9)
mini_sims = np.stack([cos_rows(p_all, o_all[o]) for o in OPTS], axis=1)
mini_top3 = np.argsort(-mini_sims, axis=1)[:, :3]

# ---- TF-IDF ranking (Milestone 1 approach) ----
from sklearn.feature_extraction.text import TfidfVectorizer
corpus = list(prompts) + [t for o in OPTS for t in ds[o]]
vec = TfidfVectorizer().fit(corpus)
P = vec.transform(prompts)
tfidf_sims = np.zeros((len(ds), 5))
for i, o in enumerate(OPTS):
    tfidf_sims[:, i] = np.asarray(P.multiply(vec.transform(ds[o])).sum(1)).ravel()
tfidf_top3 = np.argsort(-tfidf_sims, axis=1)[:, :3]

def map_at_3(top3, answers):
    tot = 0.0
    for i, a in enumerate(answers):
        gt = OPTS.index(a)
        for k in range(3):
            if top3[i][k] == gt:
                tot += 1.0/(k+1); break
    return tot/len(answers)

mini_map3 = round(map_at_3(mini_top3, answers), 4)

count = 0
for i, a in enumerate(answers):
    gt = OPTS.index(a)
    if (gt not in tfidf_top3[i]) and (gt in mini_top3[i]):
        count += 1

print("ANSWER Q10 (MiniLM MAP@3):", mini_map3)
print("ANSWER Q10 (TFIDF-miss & MiniLM-hit count):", count)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

ANSWER Q10 (MiniLM MAP@3): 0.4231
ANSWER Q10 (TFIDF-miss & MiniLM-hit count): 527


In [ ]:
from transformers import pipeline

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",
              device=0 if torch.cuda.is_available() else -1)
row = ds[1]
labels = [row["A"], row["B"], row["C"]]
res = zs(row["prompt"], candidate_labels=labels)      
ans11 = round(res["scores"][0], 4)                        
print("ANSWER Q11:", ans11)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ANSWER Q11: 0.4575


In [ ]:
res_ml = zs(row["prompt"], candidate_labels=labels, multi_label=True)  
sum_softmax = sum(res["scores"])      
sum_sigmoid = sum(res_ml["scores"])    
ans12 = round(abs(sum_softmax - sum_sigmoid), 4)
print("ANSWER Q12:", ans12)

ANSWER Q12: 0.9995


In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

r0 = ds[0]
prompt_str = (f'Question: {r0["prompt"]}. Is the correct answer '
              f'A: {r0["A"]} or B: {r0["B"]}? Answer with just the letter A or B.')

inputs = t5_tok(prompt_str, return_tensors="pt")
out_ids = t5_model.generate(**inputs, max_new_tokens=5)
answer = t5_tok.decode(out_ids[0], skip_special_tokens=True)
print("ANSWER Q13:", repr(answer))

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

ANSWER Q13: 'B'
